# TruthLayer synthetic artifact proof

This notebook verifies reproducible synthetic fusion and conformal artifacts. It is not evidence of production model performance.

In [ ]:
from pathlib import Path
import json, math
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
fusion = json.loads((ROOT / 'artifacts/fusion/synthetic-logistic-v1.json').read_text())
calibration = json.loads((ROOT / 'artifacts/conformal/synthetic-90-v1.json').read_text())
rows = [json.loads(line) for line in (ROOT / 'data/synthetic/evaluation.jsonl').read_text().splitlines()]
fusion['training'], calibration

In [ ]:
def predict(row):
    logit = fusion['bias'] + sum(weight * row['features'][name] for name, weight in zip(fusion['featureNames'], fusion['weights']))
    return 1 / (1 + math.exp(-logit))
accuracy = sum((predict(row) >= .5) == bool(row['label']) for row in rows) / len(rows)
print({'evaluation_examples': len(rows), 'synthetic_accuracy': round(accuracy, 3), 'conformal_quantile': calibration['nonconformity_quantile']})

In [ ]:
samples = ['Paris is the capital of France.', 'The capital of France is Paris.', 'Lyon is the capital of France.']
clusters = [['Paris is the capital of France.', 'The capital of France is Paris.'], ['Lyon is the capital of France.']]
probabilities = [len(cluster)/len(samples) for cluster in clusters]
semantic_entropy = -sum(p * math.log(p) for p in probabilities)
print({'semantic_entropy_demo': round(semantic_entropy, 4), 'note': 'Production clustering uses bidirectional NLI, not this manual demonstration.'})